In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

rules = [
    # Priorité 10 : établissements non bancaires et cas très précis
    (10, r"barid cash", "Barid Cash", "transfert d'argent"),
    (10, r"wafa ?cash", "Wafacash", "transfert d'argent"),
    (10, r"chaabi ?cash", "Chaabi Cash", "transfert d'argent"),
    (12, r"western union", "Western Union", "transfert d'argent"),
    (12, r"money ?gram", "MoneyGram", "transfert d'argent"),
    (10, r"bureau de change|currency ex|multichange", "Bureau de change", "change"),
    (10, r"wafa ?salaf", "Wafasalaf", "crédit à la consommation"),
    (10, r"(^| )axa( |$)", "AXA Crédit", "crédit à la consommation"),
    (10, r"wafa ?imm?eu?bilier", "Wafa Immobilier", "crédit immobilier"),
    (10, r"amana", "Al Amana Microfinance", "microcrédit"),
    (10, r"t+a?w+fi[qk]", "Attawfiq Micro-Finance", "microcrédit"),
    (10, r"canal m|كنال إم", "Canal M", "transfert d'argent"),
    (10, r"wafa assurance", "Wafa Assurance", "assurance"),
    (10, r"sofac", "Sofac", "crédit à la consommation"),
    (10, r"eqdom", "Eqdom", "crédit à la consommation"),
    (10, r"salafin", "Salafin", "crédit à la consommation"),
    (10, r"vivalis", "Vivalis", "crédit à la consommation"),
    (10, r"taslif", "Taslif", "crédit à la consommation"),
    (20, r"(^| )bti( |$)", "BTI Bank", "banque participative"),
    (15, r"micro ?(credit|finance)", "Autre microcrédit", "microcrédit"),
    # Priorité 20 : banques
    (20, r"(^| )(bank al ?magh?r[ie]b|banque du maroc|بنك المغرب)( |$)", "Bank Al-Maghrib", "banque centrale"),
    (20, r"att?ij|attir|التجاري|وفا بنك|wafa ?ban|(^| )awb( |$)", "Attijariwafa Bank", "banque"),
    (20, r"banque popula|bank popula|chaabi(?! ?cash)|البنك الشعبي|(^| )bp( |$)", "Banque Populaire", "banque"),
    (20, r"bank of africa|b ?m ?c ?e|bcme|commerce exterieur|بنك أفريقيا|بنك افريقيا|للتجارة الخارجية", "Bank of Africa", "banque"),
    (20, r"bmci|للتجارة والصناعة", "BMCI", "banque"),
    (20, r"credit agricole|cnca|القرض الفلاحي", "Crédit Agricole du Maroc", "banque"),
    (20, r"credit du maroc|مصرف المغرب", "Crédit du Maroc", "banque"),
    (20, r"(^| )(c ?i+ ?h|cin)( |$)|credit immobilier et hotelier|القرض العقاري", "CIH Bank", "banque"),
    (20, r"al ?barid|barid bank|البريد بنك|بريد بنك|poste maroc|la poste", "Al Barid Bank", "banque"),
    (20, r"saham|سهام|societe generale|s ?g ?m ?b", "Saham Bank", "banque"),
    (20, r"(^| )cfg( |$)", "CFG Bank", "banque"),
    (20, r"arab bank", "Arab Bank", "banque"),
    (20, r"assafa|الصفاء", "Bank Assafa", "banque participative"),
    (20, r"umnia|أمنية", "Umnia Bank", "banque participative"),
    (20, r"akhdar|الأخضر", "Al Akhdar Bank", "banque participative"),
    (20, r"yousr|اليسر", "Bank Al Yousr", "banque participative"),
    (20, r"dar al amane|دار الأمان", "Dar Al Amane", "banque participative"),
]
rules_df = spark.createDataFrame(
    rules, "priority int, pattern string, bank string, institution_type string"
)
rules_df.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.bank_lakehouse.ref_bank_rules")


def normalize(col):
    text = F.lower(col)
    text = F.translate(text, "éèêëàâäîïôöùûüç", "eeeeaaaiioouuuc")
    text = F.regexp_replace(text, r"[^\p{L}\p{N}]+", " ")
    return F.trim(text)


agencies = (
    spark.table("workspace.bank_lakehouse.bronze_osm_banks")
    .filter(F.col("_ingest_date") == "2026-11-06")
    .select(
        "osm_type", "osm_id",
        F.col("tags")["brand"].alias("brand"),
        F.col("tags")["name"].alias("name"),
        F.col("tags")["operator"].alias("operator"),
    )
    .withColumn("match_text", normalize(F.concat_ws(" ", "brand", "name", "operator")))
)

agency_key = Window.partitionBy("osm_type", "osm_id")
matches = agencies.join(F.broadcast(rules_df), F.expr("match_text rlike pattern"), "left")
best = (
    matches
    .withColumn("best_priority", F.min("priority").over(agency_key))
    .withColumn(
        "n_banks",
        F.size(F.collect_set(
            F.when(F.col("priority") == F.col("best_priority"), F.col("bank"))
        ).over(agency_key)),
    )
    .withColumn(
        "rn",
        F.row_number().over(
            agency_key.orderBy(F.col("priority").asc_nulls_last(), F.col("bank"))
        ),
    )
    .filter("rn = 1")
    .drop("rn", "pattern", "best_priority")
)

best.agg(
    F.count("*").alias("agences"),
    F.sum(F.col("bank").isNotNull().cast("int")).alias("identifiees"),
    F.sum((F.col("n_banks") > 1).cast("int")).alias("ambigues"),
    F.sum((F.col("match_text") == "").cast("int")).alias("sans_texte"),
).show()

In [0]:
display(
    best.filter("bank is null and match_text != ''")
        .groupBy("match_text").count()
        .orderBy(F.desc("count")).limit(50)
)

In [0]:
display(
    best.filter("n_banks > 1")
        .select("brand", "name", "operator", "match_text", "bank")
)